## Cargar variables de entorno

In [1]:
from dotenv import load_dotenv

# Load environment variables
load_dotenv(dotenv_path=".env", override=True)

True

## Crear aplicación AI 

### Setup 

Como siempre, definamos nuestro prompt y demos a nuestra aplicación acceso a la web.

In [2]:
# Inicializar herramienta de búsqueda web.
from langchain_community.tools.tavily_search import TavilySearchResults

web_search_tool = TavilySearchResults(max_results=1)

# Definir prompt template
prompt = """Sos un profesor y un experto en explicar temas complejos de una manera fácil de entender.
Tu trabajo es responder la pregunta dada de forma que incluso un niño de 5 años pueda comprenderla.
Se te ha brindado el contexto necesario para responder la pregunta.

Pregunta: {question} 

Contexto: {context}

Respuesta:"""

C:\Users\sergi\AppData\Local\Temp\ipykernel_22768\750042844.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults
C:\Users\sergi\AppData\Local\Temp\ipykernel_22768\750042844.py:4: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search_tool = TavilySearchResults(max_results=1)


### Definir la lógica de la aplicación.

La lógica acá es la misma que en el módulo de trazas. Definimos un paso de búsqueda para explorar la web y un paso de explicación para que un modelo de lenguaje resuma los resultados encontrados.

In [3]:
from openai import OpenAI
from langsmith import traceable
from langsmith.wrappers import wrap_openai


# Crear application
openai_client = wrap_openai(OpenAI())

@traceable
def search(question):
    web_docs = web_search_tool.invoke({"query": question})
    web_results = "\n".join([d["content"] for d in web_docs])
    return web_results
    
@traceable
def explain(question, context):
    formatted = prompt.format(question=question, context=context)
    
    completion = openai_client.chat.completions.create(
        messages=[
            {"role": "system", "content": formatted},
            {"role": "user", "content": question},
        ],
        model="gpt-3.5-turbo",
    )
    return completion.choices[0].message.content

@traceable
def eli5(question):
    context = search(question)
    answer = explain(question, context)
    return answer


## Setup del experimento

Ahora estamos listos para ejecutar experimentos y probar el rendimiento de nuestra aplicación sobre nuestro dataset.

### Importar cliente LangSmith 

Primero, vamos a crear un cliente de LangSmith para usar el SDK y especificar el dataset sobre el que queremos ejecutar nuestro experimento.

In [4]:
from langsmith import Client

client = Client()
#dataset_name = "eli5-silver"
#dataset_name = "ds-silver-turmeric-30"
dataset_name = "eli5-golden"

### Definir evaluadores

#### Evaluador de código personalizado

Primero definiremos un evaluador de código personalizado, que resulta útil para medir métricas deterministas o de respuesta cerrada.

In [5]:
def conciseness(outputs: dict) -> bool:
    words = outputs["output"].split(" ")
    return len(words) <= 200

Este evaluador de código personalizado es simplemente una función de Python que verifica si nuestra aplicación produce respuestas de 200 palabras o menos.

#### LLM-as-a-Judge Evaluador

Para métricas abiertas, puede ser muy potente usar un LLM para puntuar las respuestas.

Usemos un LLM para comprobar si nuestra aplicación produce resultados correctos. Primero, definamos un esquema de puntuación que nuestro LLM deba seguir en su respuesta.

In [6]:
from pydantic import BaseModel, Field

# Definir un esquema de puntuación al que nuestro LLM debe ajustarse.
class CorrectnessScore(BaseModel):
    """Correctness score of the answer when compared to the reference answer."""
    score: int = Field(description="The score of the correctness of the answer, from 0 to 1")

Vamos a definir una función para darle a un LLM las salidas de nuestra aplicación, junto con las salidas de referencia guardadas en nuestro conjunto de datos.

De este modo, el LLM podrá usar la respuesta “correcta” como referencia para juzgar si la respuesta de nuestra aplicación cumple con nuestros estándares de precisión.

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage


def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    prompt = """
    Sos un etiquetador de datos experto que evalúa las respuestas de un modelo para verificar su corrección.
Tu tarea es asignar una puntuación basada en la siguiente rúbrica:

    <Rubric>
        Una respuesta correcta:
            - Brinda información precisa
            - Usa analogías y ejemplos adecuados
            - No contiene errores fácticos
            - Es lógicamente consistente

        Al puntuar, debés penalizar:
            - Errores fácticos
            - Analogías y ejemplos incoherentes
            - Inconsistencias lógicas    
    </Rubric>

    <Instructions>
        - Leé con atención la entrada y la salida.
        - Usá la salida de referencia para determinar si la salida del modelo contiene errores.
        - Concentrate en si la salida del modelo usa analogías precisas y es lógicamente consistente.
    </Instructions>

    <Reminder>
        Las analogías de la salida no necesitan coincidir exactamente con la salida de referencia. Concentrate en la consistencia lógica.
    </Reminder>

    <input>
        {}
    </input>

    <output>
        {}
    </output>

    Usá las salidas de referencia que aparecen abajo para ayudarte a evaluar la corrección de la respuesta.
    <reference_outputs>
        {}
    </reference_outputs>
    """.format(inputs["question"], outputs["output"], reference_outputs["output"])
    structured_llm = ChatOpenAI(model_name="gpt-4o", temperature=0).with_structured_output(CorrectnessScore)
    generation = structured_llm.invoke([HumanMessage(content=prompt)])
    return generation.score == 1


### Definir Run Function

Vamos a definir una función para ejecutar nuestra aplicación sobre las entradas de ejemplo de nuestro dataset. Esta es la función que se va a llamar cuando ejecutemos nuestro experimento.

In [8]:
# Definir una función para ejecutar tu aplicación.
def run(inputs: dict):
    return eli5(inputs["question"])

#examples = list(client.list_examples(dataset_name=dataset_name, limit=3))

#for ex in examples:
#    print("INPUTS:", ex.inputs)
#    print("OUTPUTS:", ex.outputs)
#    print("---")

## Correr experimento

Tenemos todos los componentes necesarios, así que ejecutemos nuestro experimento!

In [9]:
from langsmith import evaluate

evaluate(
    run,
    data=dataset_name,
    evaluators=[correctness, conciseness],
    experiment_prefix="eli5-3.5-turbo"
)

c:\Users\sergi\OneDrive\Documentos\UCEMA\cursoGenAI-LangSmith\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'eli5-3.5-turbo-e5737c8f' at:
https://smith.langchain.com/o/11afda74-8804-4cb8-8ad4-c2a9d67d44e5/datasets/ac205640-1966-4643-b775-77cedb7777c2/compare?selectedSessions=090b4176-f92b-4742-85fd-4b6cff567006




11it [01:09,  6.34s/it]


,inputs.question,outputs.output,error,reference.output,feedback.correctness,feedback.conciseness,execution_time,example_id,id
0,cómo funciona la electricidad?,Imagina que la electricidad es como la magia q...,None,La electricidad es como una fila de bolitas in...,False,True,5.785991,d009352b-17be-4d84-b8d0-4641955c69f1,019eb9ca-984d-7ea1-9a49-f1f848be9c88
1,Qué es la biotecnología?,La biotecnología es como una varita mágica que...,None,La biotecnología es como usar la magia de la n...,True,True,5.309612,2eca8570-d5e9-4537-a707-d30c98d61ec3,019eb9ca-bc67-7a32-9c5d-2133bd67e7d7
2,How does string theory work?,"Imagine that instead of tiny points, like we u...",None,"Okay! Imagine that everything in the universe,...",True,True,5.713986,0a18be27-ed7c-475a-8ebf-30e3a6e8fbeb,019eb9ca-d40b-7790-b596-f2839ee90fc1
3,What is the Langchain framework?,Imagine LangChain as a special toolbox that he...,None,Okay! Imagine you want to build a really cool ...,False,True,4.371955,42c51fc7-bda1-491a-bf2a-e4482b32a061,019eb9ca-fa10-7732-9799-d3d816565dfc
4,What is trustcall library?,Trustcall library is like a magic wand for com...,None,"Alright, imagine you have a toy box where each...",False,True,4.459455,647650cf-0269-488b-b63a-430a4fd65d55,019eb9cb-0e22-7a10-82e2-fc2744307255
5,What is LangGraph?,LangGraph is like a special tool that helps co...,None,"Okay, imagine you have a big box of LEGO brick...",False,True,4.790888,73d89e92-2e3b-4700-8229-bde8b9f998c5,019eb9cb-246e-7b70-b41c-71cf5c8a410c
6,What is LangSmith by LangChain?,Imagínate que LangChain es como un juego de co...,None,Okay! Imagine you have a big box of toys that ...,True,True,4.167847,7d534b34-5592-45cb-863f-ba516927a103,019eb9cb-39af-7641-a179-e7316c512fcf
7,Why is the sky blue?,"Imagine you are outside on a sunny day, lookin...",None,Alright! Imagine the sky is like a big bowl of...,True,True,4.128911,9a225704-5d40-405e-8ccd-6aaa8e9fafa0,019eb9cb-4d9b-7d61-a1ac-c4e423856559
8,What is sound?,Imagine sound as tiny waves that travel throug...,None,Okay! Imagine you have a drum. When you hit it...,True,True,5.756321,9f3c2c0a-cb9d-4c06-a4b0-90958085a5e9,019eb9cb-6050-7cd1-9a12-14ebca9902fa
9,How does a democracy work?,"In a democracy, everyone gets to have a say in...",None,Okay! Imagine you and your friends want to dec...,True,True,5.260097,a1f316b5-f5e7-4358-9352-2f2372055970,019eb9cb-79cf-7270-9936-ff733bd4d184
